<a href="https://colab.research.google.com/github/HimanshiTomer/major-project/blob/main/Urban_Change_Expansion_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

soumikrakshit_onera_satellite_change_detection_dataset_path = kagglehub.dataset_download('soumikrakshit/onera-satellite-change-detection-dataset')
mdrifaturrahman33_levir_cd_path = kagglehub.dataset_download('mdrifaturrahman33/levir-cd')
himanshitomer_training_path = kagglehub.dataset_download('himanshitomer/training')

print('Data source import complete.')


In [ ]:
# Urban Expansion / Change Detection from Satellite Images
# Himanshi Tomer | BCA 6th Semester | JIMS, GGSIPU | 2023-26
#
# PLATFORM : Kaggle Notebooks (T4 GPU)
# DATASETS :
#   LEVIR-CD : kagglehub.dataset_download("mdrifaturrahman33/levir-cd")
#   OSCD     : kagglehub.dataset_download("soumikrakshit/onera-satellite-change-detection-dataset")
# NOTE: OSCD images are multi-band Sentinel-2 TIFs (B01-B13).
#       RGB is built from B04 (Red) + B03 (Green) + B02 (Blue).

#Installs and Imports

In [ ]:
import kagglehub
import os, glob, random, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm
from PIL import Image
import cv2
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from tensorflow.keras import mixed_precision

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# mixed_float16: ~2x VRAM savings, ~1.5x speed gain on T4.
# Final Conv2D output layers keep dtype="float32" for numerical stability.
mixed_precision.set_global_policy("mixed_float16")
print(f"TensorFlow  : {tf.__version__}")
print(f"GPUs        : {gpus}")
print(f"Compute dtype  : {mixed_precision.global_policy().compute_dtype}")
print(f"Variable dtype : {mixed_precision.global_policy().variable_dtype}")

# GPU Setup

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    strategy = tf.distribute.MirroredStrategy()
    print(f"GPU strategy — {strategy.num_replicas_in_sync} device(s)")
else:
    strategy = tf.distribute.get_strategy()
    print("No GPU found, using CPU")

print(f"Replicas in sync: {strategy.num_replicas_in_sync}")

import threading, time

def _keep_alive():
    while True:
        _ = tf.constant(1) + tf.constant(1)
        time.sleep(30)

_ka_thread = threading.Thread(target=_keep_alive, daemon=True)
_ka_thread.start()
print("Keep-alive thread started (GPU heartbeat every 30s)")

# Load Saved Training Outputs from Kaggle Dataset

In [ ]:
import kagglehub, shutil

# Download the saved training dataset (logs + weights + plots)
_training_dl = kagglehub.dataset_download("himanshitomer/training")
print(f"Dataset path: {_training_dl}")

# Mirror the folder structure into /kaggle/working/Output/
OUTPUT_DIR = "/kaggle/working/Output"
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")
PLOTS_DIR  = os.path.join(OUTPUT_DIR, "plots")
LOGS_DIR   = os.path.join(OUTPUT_DIR, "logs")
for d in [OUTPUT_DIR, MODELS_DIR, PLOTS_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

src_base = os.path.join(_training_dl, "Output")
for sub, dst in [("logs", LOGS_DIR), ("models", MODELS_DIR), ("plots", PLOTS_DIR)]:
    src = os.path.join(src_base, sub)
    if os.path.isdir(src):
        for fname in os.listdir(src):
            shutil.copy2(os.path.join(src, fname), os.path.join(dst, fname))
        print(f"  Copied {sub}/  → {dst}  ({len(os.listdir(dst))} files)")
    else:
        print(f"  WARNING: {src} not found")

print("\nLogs available:")
for f in sorted(os.listdir(LOGS_DIR)):   print(f"  {f}")
print("Models available:")
for f in sorted(os.listdir(MODELS_DIR)): print(f"  {f}")
print("Plots available:")
for f in sorted(os.listdir(PLOTS_DIR)):  print(f"  {f}")

#Configuration and Paths

In [ ]:
print("Downloading LEVIR-CD...")
_levir_dl   = kagglehub.dataset_download("mdrifaturrahman33/levir-cd")
print("Downloading OSCD...")
_oscd_dl    = kagglehub.dataset_download("soumikrakshit/onera-satellite-change-detection-dataset")

LEVIR_ROOT  = os.path.join(_levir_dl, "LEVIR CD")
LEVIR_TRAIN = {"t1": os.path.join(LEVIR_ROOT,"train","A"), "t2": os.path.join(LEVIR_ROOT,"train","B"), "mask": os.path.join(LEVIR_ROOT,"train","label")}
LEVIR_VAL   = {"t1": os.path.join(LEVIR_ROOT,"val","A"),   "t2": os.path.join(LEVIR_ROOT,"val","B"),   "mask": os.path.join(LEVIR_ROOT,"val","label")}
LEVIR_TEST  = {"t1": os.path.join(LEVIR_ROOT,"test","A"),  "t2": os.path.join(LEVIR_ROOT,"test","B"),  "mask": os.path.join(LEVIR_ROOT,"test","label")}

OSCD_IMAGES_ROOT = os.path.join(_oscd_dl, "images", "Onera Satellite Change Detection dataset - Images")
OSCD_LABELS_ROOT = os.path.join(_oscd_dl, "train_labels", "Onera Satellite Change Detection dataset - Train Labels")

OUTPUT_DIR = "/kaggle/working/Output"
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")
PLOTS_DIR  = os.path.join(OUTPUT_DIR, "plots")
LOGS_DIR   = os.path.join(OUTPUT_DIR, "logs")
for d in [OUTPUT_DIR, MODELS_DIR, PLOTS_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

IMG_SIZE               = 256
BATCH_SIZE_PER_REPLICA = 8
BATCH_SIZE             = BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync
EPOCHS        = 50
LEARNING_RATE = 1e-4
PATCH_SIZE    = 256
STRIDE        = 256
ENC_FILTERS   = (64, 128, 256, 512)
print(f"Batch size: {BATCH_SIZE}")

print("\nPath check:")
for label, path in [("LEVIR train/A", LEVIR_TRAIN["t1"]), ("LEVIR val/A", LEVIR_VAL["t1"]),
                     ("LEVIR test/A", LEVIR_TEST["t1"]), ("OSCD images", OSCD_IMAGES_ROOT),
                     ("OSCD labels", OSCD_LABELS_ROOT)]:
    print(f"  [{'OK' if os.path.isdir(path) else 'MISSING'}] {label}: {path}")

#Data Utilities

In [ ]:
def load_split_paths(split_dirs):
    """
    Returns matched (t1_path, t2_path, mask_path) triples from one split.
    Matches by filename stem — only files present in all three folders included.
    """
    img_exts = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

    def scan(folder):
        if not os.path.isdir(folder):
            raise FileNotFoundError(f"Directory not found: {folder}")
        found = {}
        for fname in os.listdir(folder):
            if os.path.splitext(fname)[1].lower() in img_exts:
                found[os.path.splitext(fname)[0]] = os.path.join(folder, fname)
        return found

    t1_map   = scan(split_dirs["t1"])
    t2_map   = scan(split_dirs["t2"])
    mask_map = scan(split_dirs["mask"])

    print(f"  A:{len(t1_map)}  B:{len(t2_map)}  label:{len(mask_map)}", end="")
    common = sorted(set(t1_map) & set(t2_map) & set(mask_map))
    dropped = max(len(t1_map), len(t2_map), len(mask_map)) - len(common)
    if dropped:
        print(f"  (skipped {dropped} unmatched)", end="")
    print(f"  → {len(common)} matched triplets")
    if not common:
        raise ValueError(f"No matching filenames across A, B, label: {split_dirs}")
    return [(t1_map[s], t2_map[s], mask_map[s]) for s in common]

print("Loading LEVIR-CD splits...")
train_paths = load_split_paths(LEVIR_TRAIN)
val_paths   = load_split_paths(LEVIR_VAL)
test_paths  = load_split_paths(LEVIR_TEST)
print(f"Total: train={len(train_paths)}  val={len(val_paths)}  test={len(test_paths)}")

# Quick sanity check on first image
_t1 = np.array(Image.open(train_paths[0][0]))
_mk = np.array(Image.open(train_paths[0][2]))
print(f"T1 shape={_t1.shape} dtype={_t1.dtype}")
print(f"Mask shape={_mk.shape} unique={np.unique(_mk)}")

#Data Exploratory Analysis

In [ ]:
def plot_dataset_samples(paths, n_samples=4, split_name="Train"):
    """
    n_samples random triplets, 4 columns:
    T1 | T2 | Ground Truth Mask | Change Overlay (changed pixels = red on T2)
    """
    samples = random.sample(paths, min(n_samples, len(paths)))
    fig, axes = plt.subplots(n_samples, 4, figsize=(20, n_samples * 4.5))
    if n_samples == 1:
        axes = [axes]
    fig.suptitle(f"LEVIR-CD — {split_name} Set Samples",
                 fontsize=15, fontweight="bold", y=1.01)
    col_titles = ["T1 (A/)", "T2 (B/)", "Ground Truth (label/)", "Change Overlay"]
    for ax, ct in zip(axes[0], col_titles):
        ax.set_title(ct, fontsize=10, fontweight="bold")
    for i, (t1p, t2p, mp) in enumerate(samples):
        t1 = np.array(Image.open(t1p).convert("RGB"))
        t2 = np.array(Image.open(t2p).convert("RGB"))
        mb = (np.array(Image.open(mp).convert("L")) > 127).astype(np.uint8)
        ov = t2.copy(); ov[mb == 1] = [255, 50, 50]
        axes[i][0].imshow(t1);                              axes[i][0].axis("off")
        axes[i][1].imshow(t2);                              axes[i][1].axis("off")
        axes[i][2].imshow(mb, cmap="gray", vmin=0, vmax=1); axes[i][2].axis("off")
        axes[i][3].imshow(ov);                              axes[i][3].axis("off")
        axes[i][0].set_ylabel(f"Sample {i+1}\n{100*mb.mean():.1f}% changed",
                               fontsize=9, rotation=0, labelpad=82)
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/01_samples_{split_name.lower()}.png",
                dpi=150, bbox_inches="tight")
    plt.show()

plot_dataset_samples(train_paths, n_samples=4, split_name="Train")

def compute_split_stats(paths, label, sample_size=200):
    """Returns array of per-image changed-pixel % for up to sample_size images."""
    sampled = random.sample(paths, min(sample_size, len(paths)))
    return np.array([
        100.0 * (np.array(Image.open(mp).convert("L")) > 127).mean()
        for _, _, mp in tqdm(sampled, desc=f"stats:{label}")
    ])

stats_train = compute_split_stats(train_paths, "train")
stats_val   = compute_split_stats(val_paths,   "val")
stats_test  = compute_split_stats(test_paths,  "test")

def plot_dataset_statistics(stats_train, stats_val, stats_test):
    """2x2 figure: histogram, boxplot, severity donut, summary table."""
    fig = plt.figure(figsize=(18, 11))
    fig.suptitle("LEVIR-CD Dataset Statistics", fontsize=14, fontweight="bold")
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)
    split_data   = [stats_train, stats_val, stats_test]
    split_labels = ["Train", "Val", "Test"]
    split_colors = ["#2196F3", "#FF9800", "#4CAF50"]

    ax1 = fig.add_subplot(gs[0, 0])
    for data, lbl, col in zip(split_data, split_labels, split_colors):
        ax1.hist(data, bins=25, color=col, edgecolor="white",
                 alpha=0.6, label=f"{lbl} (mean={data.mean():.1f}%)")
    ax1.set_xlabel("% Area Changed"); ax1.set_ylabel("Image Count")
    ax1.set_title("Change Distribution Per Split")
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2 = fig.add_subplot(gs[0, 1])
    bp = ax2.boxplot(split_data, patch_artist=True, vert=True, widths=0.5,
                     medianprops=dict(color="black", linewidth=2.5),
                     flierprops=dict(marker="o", alpha=0.35))
    for patch, col in zip(bp["boxes"], split_colors):
        patch.set_facecolor(col); patch.set_alpha(0.75)
    ax2.set_xticks([1, 2, 3]); ax2.set_xticklabels(split_labels, fontsize=12)
    ax2.set_ylabel("% Changed"); ax2.set_title("Spread Per Split")
    ax2.grid(True, alpha=0.3, axis="y")

    ax3 = fig.add_subplot(gs[1, 0])
    ax3.pie(
        [(stats_train < 5).sum(),
         ((stats_train >= 5)  & (stats_train < 15)).sum(),
         ((stats_train >= 15) & (stats_train < 30)).sum(),
         (stats_train >= 30).sum()],
        labels=["Minimal (<5%)", "Low (5-15%)", "High (15-30%)", "Severe (>30%)"],
        colors=["#4CAF50", "#FFC107", "#FF5722", "#B71C1C"],
        autopct="%1.1f%%", startangle=90,
        wedgeprops=dict(edgecolor="white", linewidth=2, width=0.65)
    )
    ax3.set_title("Train Set — Change Severity")

    ax4 = fig.add_subplot(gs[1, 1])
    ax4.axis("off")
    all_data = np.concatenate(split_data)
    table_rows = [
        ["Split",         "Train",                           "Val",                         "Test"],
        ["Image count",   str(len(train_paths)),             str(len(val_paths)),            str(len(test_paths))],
        ["Sampled",       str(len(stats_train)),             str(len(stats_val)),            str(len(stats_test))],
        ["Mean change",   f"{stats_train.mean():.2f}%",     f"{stats_val.mean():.2f}%",     f"{stats_test.mean():.2f}%"],
        ["Median change", f"{np.median(stats_train):.2f}%", f"{np.median(stats_val):.2f}%", f"{np.median(stats_test):.2f}%"],
        ["Std",           f"{stats_train.std():.2f}%",      f"{stats_val.std():.2f}%",      f"{stats_test.std():.2f}%"],
        ["Max",           f"{stats_train.max():.2f}%",      f"{stats_val.max():.2f}%",      f"{stats_test.max():.2f}%"],
        ["Overall mean",  f"{all_data.mean():.2f}%",        "—",                            "—"],
    ]
    tbl = ax4.table(
        cellText=[r[1:] for r in table_rows[1:]],
        rowLabels=[r[0] for r in table_rows[1:]],
        colLabels=table_rows[0][1:],
        cellLoc="center", loc="center", bbox=[0.0, 0.0, 1.0, 1.0]
    )
    tbl.auto_set_font_size(False); tbl.set_fontsize(11)
    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor("#1565C0")
            cell.set_text_props(color="white", fontweight="bold")
        elif row % 2 == 0:
            cell.set_facecolor("#E3F2FD")
    ax4.set_title("Per-Split Summary Statistics", fontsize=12, fontweight="bold", pad=10)
    plt.savefig(f"{PLOTS_DIR}/02_dataset_statistics.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_dataset_statistics(stats_train, stats_val, stats_test)

fig, ax = plt.subplots(figsize=(9, 5))
sizes  = [len(train_paths), len(val_paths), len(test_paths)]
labels = [f"Train\n({sizes[0]})", f"Val\n({sizes[1]})", f"Test\n({sizes[2]})"]
bars   = ax.bar(labels, sizes, color=["#2196F3", "#FF9800", "#4CAF50"],
                edgecolor="white", linewidth=2, width=0.5)
for bar, s in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(s), ha="center", fontweight="bold", fontsize=13)
ax.set_title("LEVIR-CD Split Sizes")
ax.set_ylabel("Image Pairs"); ax.set_ylim(0, max(sizes) * 1.18)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/03_split_sizes.png", dpi=150, bbox_inches="tight")
plt.show()

# Data Pipeline — Streaming from Disk

Patches are generated on-the-fly using a `tf.data` generator.
No full dataset is loaded into RAM — each image is opened, tiled, and fed
directly into the training pipeline.

In [ ]:
n_train = len(train_paths) * (IMG_SIZE // PATCH_SIZE) ** 2
n_val   = len(val_paths)   * (IMG_SIZE // PATCH_SIZE) ** 2
n_test  = len(test_paths)  * (IMG_SIZE // PATCH_SIZE) ** 2
print(f"Estimated patch counts  train={n_train:,}  val={n_val:,}  test={n_test:,}")
print("Patches streamed from disk — no bulk RAM allocation.")

fig, ax = plt.subplots(figsize=(9, 5))
p_sizes = [n_train, n_val, n_test]
bars = ax.bar([f"Train\n({p_sizes[0]:,})", f"Val\n({p_sizes[1]:,})", f"Test\n({p_sizes[2]:,})"],
              p_sizes, color=["#2196F3","#FF9800","#4CAF50"], edgecolor="white", linewidth=2, width=0.5)
for bar, s in zip(bars, p_sizes):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10, f"{s:,}", ha="center", fontweight="bold", fontsize=12)
ax.set_title(f"Estimated Patch Counts (total {sum(p_sizes):,}) — streamed from disk")
ax.set_ylabel("Patches"); ax.set_ylim(0, max(p_sizes)*1.18); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/04_patch_counts.png", dpi=150, bbox_inches="tight")
plt.show()

# Data Pipeline

Each image triplet is opened inside a `tf.data` generator, tiled into
256x256 patches, and fed batch-by-batch. RAM stays flat regardless of dataset size.

In [ ]:
PATCH_SIZE_TF = tf.constant(PATCH_SIZE, dtype=tf.int32)
STRIDE_TF     = tf.constant(STRIDE,     dtype=tf.int32)

def _load_and_tile(t1p, t2p, mp):
    def _read_rgb(path):
        raw = tf.io.read_file(path)
        img = tf.image.decode_image(raw, channels=3, expand_animations=False)
        return tf.cast(img, tf.float32) / 255.0
    def _read_mask(path):
        raw = tf.io.read_file(path)
        img = tf.image.decode_image(raw, channels=1, expand_animations=False)
        return tf.cast(img > 127, tf.float32)

    t1 = _read_rgb(t1p); t2 = _read_rgb(t2p); mask = _read_mask(mp)
    h = tf.shape(t1)[0]; w = tf.shape(t1)[1]
    ys = tf.range(0, h - PATCH_SIZE_TF + 1, STRIDE_TF)
    xs = tf.range(0, w - PATCH_SIZE_TF + 1, STRIDE_TF)
    gy, gx = tf.meshgrid(ys, xs, indexing="ij")
    gy = tf.reshape(gy, [-1]); gx = tf.reshape(gx, [-1])

    def extract(idx):
        y = gy[idx]; x = gx[idx]
        p1 = tf.ensure_shape(t1  [y:y+PATCH_SIZE, x:x+PATCH_SIZE, :], [PATCH_SIZE, PATCH_SIZE, 3])
        p2 = tf.ensure_shape(t2  [y:y+PATCH_SIZE, x:x+PATCH_SIZE, :], [PATCH_SIZE, PATCH_SIZE, 3])
        pm = tf.ensure_shape(mask[y:y+PATCH_SIZE, x:x+PATCH_SIZE, :], [PATCH_SIZE, PATCH_SIZE, 1])
        return p1, p2, pm

    patches = tf.map_fn(extract, tf.range(tf.shape(gy)[0]),
                        fn_output_signature=(
                            tf.TensorSpec([PATCH_SIZE, PATCH_SIZE, 3], tf.float32),
                            tf.TensorSpec([PATCH_SIZE, PATCH_SIZE, 3], tf.float32),
                            tf.TensorSpec([PATCH_SIZE, PATCH_SIZE, 1], tf.float32),
                        ))
    return tf.data.Dataset.from_tensor_slices(patches)


def augment_tf(t1, t2, mask):
    if tf.random.uniform(()) > 0.5:
        t1 = tf.image.flip_left_right(t1); t2 = tf.image.flip_left_right(t2); mask = tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        t1 = tf.image.flip_up_down(t1); t2 = tf.image.flip_up_down(t2); mask = tf.image.flip_up_down(mask)
    t1 = tf.clip_by_value(tf.image.random_brightness(t1, 0.15), 0.0, 1.0)
    t2 = tf.clip_by_value(tf.image.random_brightness(t2, 0.15), 0.0, 1.0)
    t1 = tf.clip_by_value(tf.image.random_contrast(t1, 0.85, 1.15), 0.0, 1.0)
    t2 = tf.clip_by_value(tf.image.random_contrast(t2, 0.85, 1.15), 0.0, 1.0)
    return t1, t2, mask


def build_path_ds(path_list):
    return tf.data.Dataset.from_tensor_slices(
        ([p[0] for p in path_list], [p[1] for p in path_list], [p[2] for p in path_list]))


def build_fused_ds(path_list, batch_size, augment_data=False, shuffle=True):
    ds = build_path_ds(path_list)
    if shuffle: ds = ds.shuffle(len(path_list), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.flat_map(_load_and_tile)
    if augment_data: ds = ds.map(augment_tf, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda t1, t2, m: (tf.concat([t1, t2], axis=-1), m), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)


def build_siamese_ds(path_list, batch_size, augment_data=False, shuffle=True):
    ds = build_path_ds(path_list)
    if shuffle: ds = ds.shuffle(len(path_list), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.flat_map(_load_and_tile)
    if augment_data: ds = ds.map(augment_tf, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda t1, t2, m: ({"t1_input": t1, "t2_input": t2}, m), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)


train_ds_fused = build_fused_ds(train_paths, BATCH_SIZE, augment_data=True)
val_ds_fused   = build_fused_ds(val_paths,   BATCH_SIZE)
test_ds_fused  = build_fused_ds(test_paths,  BATCH_SIZE, shuffle=False)
train_ds_siam  = build_siamese_ds(train_paths, BATCH_SIZE, augment_data=True)
val_ds_siam    = build_siamese_ds(val_paths,   BATCH_SIZE)
test_ds_siam   = build_siamese_ds(test_paths,  BATCH_SIZE, shuffle=False)

for batch in train_ds_fused.take(1):
    x, y = batch
    print(f"Fused batch   X:{x.shape}  Y:{y.shape}  dtype:{x.dtype}")
for batch in train_ds_siam.take(1):
    x, y = batch
    print(f"Siamese batch t1:{x['t1_input'].shape}  Y:{y.shape}")

#Loss, Metrics and Building Blocks

In [ ]:
def dice_loss(y_true, y_pred, smooth=1e-6):
    """Dice Loss = 1 - 2|X∩Y| / (|X| + |Y|)."""
    yt = K.flatten(tf.cast(y_true, tf.float32))
    yp = K.flatten(tf.cast(y_pred, tf.float32))
    return 1.0 - (2.0 * K.sum(yt * yp) + smooth) / (K.sum(yt) + K.sum(yp) + smooth)


def bce_dice_loss(y_true, y_pred):
    """BCE (pixel-level) + Dice (region-level) combined loss."""
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return tf.reduce_mean(bce) + dice_loss(y_true, y_pred)


class IoUMetric(tf.keras.metrics.Metric):
    """IoU = TP / (TP + FP + FN). Accumulated across all batches in epoch."""
    def __init__(self, threshold=0.5, name="iou", **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold    = threshold
        self.intersection = self.add_weight(name="intersection", shape=(), initializer="zeros")
        self.union        = self.add_weight(name="union",        shape=(), initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        yp    = tf.cast(y_pred > self.threshold, tf.float32)
        yt    = tf.reshape(tf.cast(y_true, tf.float32), [-1])
        yp    = tf.reshape(yp, [-1])
        inter = tf.reduce_sum(yt * yp)
        union = tf.reduce_sum(yt) + tf.reduce_sum(yp) - inter
        self.intersection.assign_add(inter)
        self.union.assign_add(union + 1e-6)

    def result(self):
        return self.intersection / self.union

    def reset_state(self):
        self.intersection.assign(0.0); self.union.assign(0.0)


def get_metrics():
    return [
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        IoUMetric(name="iou"),
    ]


def conv_block(x, filters, batch_norm=True):
    """2x [Conv2D -> BN -> ReLU]. Shared by all three model builders."""
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding="same", kernel_initializer="he_normal")(x)
        if batch_norm:
            x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    return x

#MODEL-1 : Baseline U-Net

Input  : (H, W, 6) — T1 and T2 concatenated along channel axis (early fusion).
Output : (H, W, 1) sigmoid change probability map.

In [ ]:
def build_unet(input_channels=6, img_size=256, enc_filters=None):
    if enc_filters is None:
        enc_filters = ENC_FILTERS
    inp = layers.Input(shape=(img_size, img_size, input_channels), name="concat_input")
    skips = []; x = inp
    for f in enc_filters:
        x = conv_block(x, f); skips.append(x); x = layers.MaxPooling2D(2)(x)
    x = conv_block(x, enc_filters[-1] * 2)
    for f, skip in zip(reversed(enc_filters), reversed(skips)):
        x = layers.Conv2DTranspose(f, 2, strides=2, padding="same")(x)
        x = layers.Concatenate()([x, skip]); x = conv_block(x, f)
    return Model(inp, layers.Conv2D(1, 1, activation="sigmoid", dtype="float32", name="output")(x),
                 name="Baseline_UNet")

# model instantiated inside strategy.scope() in the Compile cell below

#MODEL-2 : DenseNet Encoder + U-Net Decoder

Dense Blocks: each layer receives concatenated outputs of all preceding layers.
Encoder = Dense Blocks + transition_down. Decoder = standard U-Net.

In [ ]:
def dense_block(x, n_layers=4, growth_rate=32):
    """Each layer appends growth_rate channels to the running feature concatenation."""
    feature_list = [x]
    for _ in range(n_layers):
        x_in = layers.Concatenate()(feature_list) if len(feature_list) > 1 else feature_list[0]
        h = layers.BatchNormalization()(x_in)
        h = layers.Activation("relu")(h)
        h = layers.Conv2D(growth_rate, 3, padding="same", kernel_initializer="he_normal")(h)
        feature_list.append(h)
    return layers.Concatenate()(feature_list)


def transition_down(x, out_filters):
    """BN -> ReLU -> 1x1 Conv -> AvgPool2D."""
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Conv2D(out_filters, 1, padding="same")(x)
    return layers.AveragePooling2D(2)(x)


def build_densenet_unet(input_channels=6, img_size=256, enc_filters=None):
    if enc_filters is None:
        enc_filters = ENC_FILTERS
    inp = layers.Input(shape=(img_size, img_size, input_channels), name="concat_input")
    skips = []
    x = layers.Conv2D(64, 7, padding="same", kernel_initializer="he_normal")(inp)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    for i, f in enumerate(enc_filters):
        x = dense_block(x, n_layers=4, growth_rate=32)
        x = layers.Conv2D(f, 1, padding="same")(x)
        x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
        skips.append(x)
        if i < len(enc_filters) - 1:
            x = transition_down(x, f)
    x = dense_block(x, n_layers=4, growth_rate=32)
    x = layers.Conv2D(enc_filters[-1] * 2, 1, padding="same")(x)
    for f_dec, skip_dec in zip(reversed(enc_filters[:-1]), reversed(skips[:-1])):
        x = layers.Conv2DTranspose(f_dec, 2, strides=2, padding="same")(x)
        x = layers.Concatenate()([x, skip_dec]); x = conv_block(x, f_dec)
    return Model(inp, layers.Conv2D(1, 1, activation="sigmoid", dtype="float32", name="output")(x),
                 name="DenseNet_UNet")

# model instantiated inside strategy.scope() in the Compile cell below

#MODEL-3 : Siamese U-Net

Shared-weight encoder processes T1 and T2 independently.
Change signal = |feat_T1 - feat_T2| at bottleneck and each skip connection.

In [ ]:
def build_siamese_unet(img_size=256, enc_filters=None):
    if enc_filters is None:
        enc_filters = ENC_FILTERS
    enc_in = layers.Input(shape=(img_size, img_size, 3), name="enc_in")
    skips_ = []; ex = enc_in
    for f in enc_filters:
        ex = conv_block(ex, f); skips_.append(ex); ex = layers.MaxPooling2D(2)(ex)
    ex = conv_block(ex, enc_filters[-1] * 2)
    shared_encoder = Model(enc_in, [ex] + skips_, name="shared_encoder")

    t1_input = layers.Input(shape=(img_size, img_size, 3), name="t1_input")
    t2_input = layers.Input(shape=(img_size, img_size, 3), name="t2_input")

    t1_outs = shared_encoder(t1_input); t2_outs = shared_encoder(t2_input)
    t1_bot  = t1_outs[0]; t2_bot = t2_outs[0]
    t1_skips = [t1_outs[k] for k in range(1, len(enc_filters) + 1)]
    t2_skips = [t2_outs[k] for k in range(1, len(enc_filters) + 1)]

    diff_bot = layers.Lambda(lambda v: tf.abs(v[0] - v[1]),
                             name="bottleneck_diff")([t1_bot, t2_bot])
    x = layers.Concatenate(name="bottleneck_fuse")([t1_bot, t2_bot, diff_bot])
    x = conv_block(x, enc_filters[-1] * 2)

    dec_filters = list(reversed(enc_filters))
    for i, f in enumerate(dec_filters):
        s1 = t1_skips[-(i + 1)]; s2 = t2_skips[-(i + 1)]
        x      = layers.Conv2DTranspose(f, 2, strides=2, padding="same")(x)
        d_skip = layers.Lambda(lambda v: tf.abs(v[0] - v[1]),
                               name=f"skip_diff_{i}")([s1, s2])
        x      = layers.Concatenate(name=f"skip_cat_{i}")([x, s1, s2, d_skip])
        x      = conv_block(x, f)

    return Model(inputs=[t1_input, t2_input],
                 outputs=layers.Conv2D(1, 1, activation="sigmoid", dtype="float32", name="output")(x),
                 name="Siamese_UNet")

# model instantiated inside strategy.scope() in the Compile cell below

#Compile All Models

In [ ]:
with strategy.scope():
    model_unet    = build_unet()
    model_dense   = build_densenet_unet()
    model_siamese = build_siamese_unet()

    def compile_model(m):
        m.compile(
            optimizer=Adam(learning_rate=LEARNING_RATE),
            loss=bce_dice_loss,
            metrics=[
                tf.keras.metrics.BinaryAccuracy(name="accuracy"),
                tf.keras.metrics.Precision(name="precision"),
                tf.keras.metrics.Recall(name="recall"),
                IoUMetric(name="iou"),
            ]
        )
        return m

    model_unet    = compile_model(model_unet)
    model_dense   = compile_model(model_dense)
    model_siamese = compile_model(model_siamese)

model_unet.summary(line_length=90)
model_dense.summary(line_length=90)
model_siamese.summary(line_length=90)
for m in [model_unet, model_dense, model_siamese]:
    print(f"{m.name}: {m.count_params():,} params")

MODEL_LABELS = ["Baseline U-Net", "DenseNet+UNet", "Siamese U-Net"]
COLORS = {"Baseline U-Net": "#2196F3", "DenseNet+UNet": "#FF5722", "Siamese U-Net": "#4CAF50"}
param_counts = [m.count_params() for m in [model_unet, model_dense, model_siamese]]

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.barh(MODEL_LABELS, [p/1e6 for p in param_counts],
               color=list(COLORS.values()), edgecolor="white", height=0.45)
for bar, p in zip(bars, param_counts):
    ax.text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
            f"{p/1e6:.2f}M", va="center", fontweight="bold", fontsize=11)
ax.set_xlabel("Parameters (Millions)"); ax.set_title("Model Parameter Counts")
ax.set_xlim(0, max(p/1e6 for p in param_counts)*1.35); ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/05_model_complexity.png", dpi=150, bbox_inches="tight")
plt.show()

#Training

In [ ]:
def get_callbacks(model_name):
    return [
        ModelCheckpoint(os.path.join(MODELS_DIR, f"{model_name}_best.weights.h5"),
                        monitor="val_iou", mode="max", save_best_only=True,
                        save_weights_only=True, verbose=1),
        EarlyStopping(monitor="val_iou", mode="max", patience=8,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-7, verbose=1),
        CSVLogger(os.path.join(LOGS_DIR, f"{model_name}_log.csv")),
    ]

print("=" * 55 + "\nTRAINING: Baseline U-Net\n" + "=" * 55)
history_unet = model_unet.fit(
    train_ds_fused, validation_data=val_ds_fused,
    epochs=EPOCHS, callbacks=get_callbacks("baseline_unet"), verbose=1)
tf.keras.backend.clear_session()
import gc; gc.collect()

print("=" * 55 + "\nTRAINING: DenseNet + U-Net\n" + "=" * 55)
history_dense = model_dense.fit(
    train_ds_fused, validation_data=val_ds_fused,
    epochs=EPOCHS, callbacks=get_callbacks("densenet_unet"), verbose=1)
tf.keras.backend.clear_session(); gc.collect()

print("=" * 55 + "\nTRAINING: Siamese U-Net\n" + "=" * 55)
history_siamese = model_siamese.fit(
    train_ds_siam, validation_data=val_ds_siam,
    epochs=EPOCHS, callbacks=get_callbacks("siamese_unet"), verbose=1)

# Training Curves — Loaded from CSV Logs

In [ ]:
LOG_FILES = {
    "Baseline U-Net": os.path.join(LOGS_DIR, "baseline_unet_log.csv"),
    "DenseNet+UNet":  os.path.join(LOGS_DIR, "densenet_unet_log.csv"),
    "Siamese U-Net":  os.path.join(LOGS_DIR, "siamese_unet_log.csv"),
}

logs = {}
for name, path in LOG_FILES.items():
    if os.path.exists(path):
        logs[name] = pd.read_csv(path)
        print(f"  Loaded {name}: {len(logs[name])} epochs")
    else:
        print(f"  MISSING: {path}")

METRIC_INFO = {
    "loss":      ("Loss",      "BCE + Dice"),
    "accuracy":  ("Accuracy",  "% pixels correctly classified"),
    "precision": ("Precision", "TP / (TP + FP)"),
    "recall":    ("Recall",    "TP / (TP + FN)"),
    "iou":       ("IoU",       "TP / (TP + FP + FN)"),
}

def plot_training_curve(metric_key, save_index):
    label, subtitle = METRIC_INFO[metric_key]
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle(f"{label} — {subtitle}", fontsize=13, fontweight="bold", y=1.02)
    best_fn = min if "loss" in metric_key else max
    for ax, (col_key, phase) in zip(axes, [(metric_key, "Training"), (f"val_{metric_key}", "Validation")]):
        ax.set_title(phase, fontsize=12)
        for name, df in logs.items():
            if col_key not in df.columns: continue
            vals = df[col_key].dropna().tolist()
            ax.plot(range(1, len(vals)+1), vals, color=COLORS[name], linewidth=2.5,
                    label=name, marker="o", markersize=2.5, alpha=0.9)
            bv = best_fn(vals); bep = vals.index(bv) + 1
            ax.scatter(bep, bv, color=COLORS[name], s=90, zorder=5)
            ax.annotate(f" {bv:.3f}", (bep, bv), fontsize=8.5, color=COLORS[name], va="bottom")
        ax.set_xlabel("Epoch"); ax.set_ylabel(label)
        ax.legend(fontsize=10); ax.grid(True, alpha=0.3, linestyle="--"); ax.set_xlim(left=1)
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/06_{save_index:02d}_curve_{metric_key}.png", dpi=150, bbox_inches="tight")
    plt.show()

for idx, mk in enumerate(["loss", "accuracy", "precision", "recall", "iou"]):
    plot_training_curve(mk, idx+1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("F1 Score — 2PR / (P+R)", fontsize=13, fontweight="bold", y=1.02)
for ax, prefix in zip(axes, ["", "val_"]):
    ax.set_title("Training" if prefix == "" else "Validation", fontsize=12)
    for name, df in logs.items():
        pc = f"{prefix}precision"; rc = f"{prefix}recall"
        if pc not in df.columns or rc not in df.columns: continue
        p = df[pc].dropna().values; r = df[rc].dropna().values
        f1 = 2 * p * r / (p + r + 1e-7)
        ax.plot(range(1, len(f1)+1), f1, color=COLORS[name],
                linewidth=2.5, label=name, marker="o", markersize=2.5)
    ax.set_xlabel("Epoch"); ax.set_ylabel("F1")
    ax.legend(); ax.grid(True, alpha=0.3, linestyle="--"); ax.set_xlim(left=1)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/06_06_curve_f1.png", dpi=150, bbox_inches="tight")
plt.show()

#Test Set Evaluation

In [ ]:
with strategy.scope():
    model_unet    = build_unet();    model_unet    = compile_model(model_unet)
    model_dense   = build_densenet_unet(); model_dense   = compile_model(model_dense)
    model_siamese = build_siamese_unet(); model_siamese = compile_model(model_siamese)

for model, name in [(model_unet,"baseline_unet"),(model_dense,"densenet_unet"),(model_siamese,"siamese_unet")]:
    wpath = os.path.join(MODELS_DIR, f"{name}_best.weights.h5")
    if os.path.exists(wpath):
        model.load_weights(wpath); print(f"Loaded: {wpath}")
    else:
        print(f"WARNING: not found: {wpath}")

MODEL_LABELS = ["Baseline U-Net", "DenseNet+UNet", "Siamese U-Net"]
COLORS = {"Baseline U-Net": "#2196F3", "DenseNet+UNet": "#FF5722", "Siamese U-Net": "#4CAF50"}

def evaluate_model(model, test_ds, name, is_siamese=False):
    acc_m  = tf.keras.metrics.BinaryAccuracy(name="accuracy")
    prec_m = tf.keras.metrics.Precision(name="precision")
    rec_m  = tf.keras.metrics.Recall(name="recall")
    iou_m  = IoUMetric(name="iou")
    losses = []

    for batch in tqdm(test_ds, desc=name):
        x, y_true = batch
        y_pred = tf.cast(model(x, training=False), tf.float32)
        y_true = tf.cast(y_true, tf.float32)
        losses.append(float(bce_dice_loss(y_true, y_pred)))
        acc_m.update_state(y_true, y_pred)
        prec_m.update_state(y_true, y_pred)
        rec_m.update_state(y_true, y_pred)
        iou_m.update_state(y_true, y_pred)

    p = float(prec_m.result()); r = float(rec_m.result())
    raw = {
        "loss":      float(np.mean(losses)),
        "accuracy":  float(acc_m.result()),
        "precision": p,
        "recall":    r,
        "f1":        2 * p * r / (p + r + 1e-7),
        "iou":       float(iou_m.result()),
    }
    print(f"\n{name}")
    for k in ["loss","accuracy","precision","recall","f1","iou"]:
        print(f"  {k:<12}: {raw[k]:.4f}  {'|'*int(raw[k]*35)}")
    return raw

results_unet    = evaluate_model(model_unet,    test_ds_fused, "Baseline U-Net")
results_dense   = evaluate_model(model_dense,   test_ds_fused, "DenseNet+UNet")
results_siamese = evaluate_model(model_siamese, test_ds_siam,  "Siamese U-Net")

results_unet    = evaluate_model(model_unet,    test_ds_fused, "Baseline U-Net")
results_dense   = evaluate_model(model_dense,   test_ds_fused, "DenseNet+UNet")
results_siamese = evaluate_model(model_siamese, test_ds_siam,  "Siamese U-Net")

ALL_RESULTS = [results_unet, results_dense, results_siamese]
METRIC_KEYS = ["accuracy","precision","recall","f1","iou"]
METRIC_LBLS = ["Accuracy","Precision","Recall","F1 Score","IoU"]

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(METRIC_KEYS)); w = 0.25
for i, (res, name) in enumerate(zip(ALL_RESULTS, MODEL_LABELS)):
    vals = [res.get(k,0) for k in METRIC_KEYS]
    bars = ax.bar(x+i*w, vals, w, label=name, color=list(COLORS.values())[i], alpha=0.85, edgecolor="white")
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.007,
                f"{v:.3f}", ha="center", va="bottom", fontsize=8.5, fontweight="bold")
ax.set_xticks(x+w); ax.set_xticklabels(METRIC_LBLS, fontsize=12)
ax.set_ylim(0,1.18); ax.set_ylabel("Score"); ax.set_title("Model Comparison — Test Set Metrics")
ax.legend(fontsize=11); ax.grid(axis="y", alpha=0.3, linestyle="--")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/07_metrics_grouped_bar.png", dpi=150, bbox_inches="tight"); plt.show()

angles = np.linspace(0, 2*np.pi, len(METRIC_KEYS), endpoint=False).tolist() + [0]
fig, ax = plt.subplots(figsize=(8,8), subplot_kw=dict(polar=True))
for res, name, color in zip(ALL_RESULTS, MODEL_LABELS, COLORS.values()):
    vals = [res.get(k,0) for k in METRIC_KEYS] + [res.get(METRIC_KEYS[0],0)]
    ax.plot(angles, vals, color=color, linewidth=2.5, label=name)
    ax.fill(angles, vals, color=color, alpha=0.12)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(METRIC_LBLS, fontsize=12, fontweight="bold")
ax.set_ylim(0,1); ax.set_yticks([0.25,0.5,0.75,1.0]); ax.set_yticklabels(["0.25","0.50","0.75","1.00"],fontsize=8)
ax.grid(True,alpha=0.35); ax.set_title("Radar — Test Set Metrics",fontsize=13,fontweight="bold",pad=25)
ax.legend(loc="upper right", bbox_to_anchor=(1.35,1.12), fontsize=11)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/08_metrics_radar.png", dpi=150, bbox_inches="tight"); plt.show()

#Prediction Visualisation

In [ ]:
def get_test_predictions(model, is_siamese=False, n=6):
    idxs  = random.sample(range(len(test_paths)), min(n, len(test_paths)))
    t1s, t2s, gts, preds_list = [], [], [], []
    for i in idxs:
        t1p, t2p, mp = test_paths[i]
        t1   = np.array(Image.open(t1p).convert("RGB"), dtype=np.float32) / 255.0
        t2   = np.array(Image.open(t2p).convert("RGB"), dtype=np.float32) / 255.0
        mask = (np.array(Image.open(mp).convert("L"), dtype=np.float32) > 127).astype(np.float32)
        h, w = t1.shape[:2]
        cy, cx = h//2, w//2; hs = PATCH_SIZE//2
        if h < PATCH_SIZE or w < PATCH_SIZE: continue
        t1c = t1[cy-hs:cy+hs, cx-hs:cx+hs]; t2c = t2[cy-hs:cy+hs, cx-hs:cx+hs]
        mc  = mask[cy-hs:cy+hs, cx-hs:cx+hs]
        t1b = t1c[np.newaxis]; t2b = t2c[np.newaxis]
        if is_siamese:
            pred = model.predict({"t1_input": t1b, "t2_input": t2b}, verbose=0, batch_size=1)
        else:
            pred = model.predict(np.concatenate([t1b, t2b], axis=-1), verbose=0, batch_size=1)
        t1s.append(t1c); t2s.append(t2c); gts.append(mc[...,np.newaxis]); preds_list.append(pred[0])
    return (np.array(t1s), np.array(t2s),
            np.array(gts), np.array(preds_list))


def plot_predictions(model, model_name, is_siamese=False, n_samples=5):
    t1s, t2s, gts, preds = get_test_predictions(model, is_siamese, n_samples)
    fig, axes = plt.subplots(len(t1s), 5, figsize=(23, len(t1s)*4.5))
    if len(t1s)==1: axes = [axes]
    fig.suptitle(f"{model_name} — Test Set Predictions", fontsize=15, fontweight="bold", y=1.01)
    for ax, ct in zip(axes[0], ["T1 (A/)","T2 (B/)","Ground Truth","Predicted (≥0.5)","Overlay"]):
        ax.set_title(ct, fontsize=10, fontweight="bold")
    for i in range(len(t1s)):
        gt    = gts[i,...,0]
        predb = (preds[i,...,0] > 0.5).astype(np.float32)
        ov    = t2s[i].copy(); ov[predb==1] = [1.0, 0.15, 0.15]
        inter = np.sum(gt*predb); ious = inter/(np.sum(gt)+np.sum(predb)-inter+1e-6)
        ps = inter/(np.sum(predb)+1e-6); rs = inter/(np.sum(gt)+1e-6); f1s = 2*ps*rs/(ps+rs+1e-6)
        for j, (img, cmap) in enumerate([(t1s[i],None),(t2s[i],None),(gt,"gray"),(predb,"gray"),(ov,None)]):
            axes[i][j].imshow(img, cmap=cmap, vmin=0 if cmap else None, vmax=1 if cmap else None)
            axes[i][j].axis("off")
        axes[i][0].set_ylabel(f"S{i+1}  IoU:{ious:.2f}  F1:{f1s:.2f}",
                               fontsize=9, rotation=0, labelpad=75)
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/09_predictions_{model_name.replace(' ','_')}.png", dpi=150, bbox_inches="tight")
    plt.show()

for model, name, siam in [(model_unet,"Baseline U-Net",False),
                           (model_dense,"DenseNet+UNet",False),
                           (model_siamese,"Siamese U-Net",True)]:
    plot_predictions(model, name, siam)

#Change Probability Heatmaps

In [ ]:
def plot_heatmaps(model, model_name, is_siamese=False, n_samples=3):
    t1s, t2s, gts, preds = get_test_predictions(model, is_siamese, n_samples)
    fig, axes = plt.subplots(len(t1s), 4, figsize=(20, len(t1s)*5))
    if len(t1s)==1: axes = [axes]
    fig.suptitle(f"{model_name} — Change Probability Heatmaps", fontsize=12, fontweight="bold", y=1.02)
    for ax, ct in zip(axes[0], ["T2 (B/)","Ground Truth","P(change) heatmap","Binary (≥0.5)"]):
        ax.set_title(ct, fontsize=10, fontweight="bold")
    for i in range(len(t1s)):
        prob  = preds[i,...,0]; predb = (prob > 0.5).astype(np.float32); gt = gts[i,...,0]
        axes[i][0].imshow(t2s[i]); axes[i][0].axis("off")
        axes[i][1].imshow(gt, cmap="gray", vmin=0, vmax=1); axes[i][1].axis("off")
        im = axes[i][2].imshow(prob, cmap="RdYlGn_r", vmin=0, vmax=1); axes[i][2].axis("off")
        plt.colorbar(im, ax=axes[i][2], fraction=0.046, pad=0.04, label="P(change)")
        axes[i][3].imshow(predb, cmap="gray", vmin=0, vmax=1); axes[i][3].axis("off")
        ious = np.sum(gt*predb)/(np.sum(gt)+np.sum(predb)-np.sum(gt*predb)+1e-6)
        axes[i][0].set_ylabel(f"S{i+1}  IoU:{ious:.2f}", fontsize=9, rotation=0, labelpad=60)
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/10_heatmap_{model_name.replace(' ','_')}.png", dpi=150, bbox_inches="tight")
    plt.show()

for model, name, siam in [(model_unet,"Baseline U-Net",False),
                           (model_dense,"DenseNet+UNet",False),
                           (model_siamese,"Siamese U-Net",True)]:
    plot_heatmaps(model, name, siam)

#Confusion Matrix

Pixel-level binary classification: Changed (1) vs No Change (0).
Left = raw counts | Right = row-normalised %.

In [ ]:
def plot_confusion_matrix(model, model_name, is_siamese=False, n_samples=400):
    idxs  = random.sample(range(len(test_paths)), min(n_samples, len(test_paths)))
    y_true_all, y_pred_all = [], []
    for i in tqdm(idxs, desc=f"CM:{model_name}"):
        t1p, t2p, mp = test_paths[i]
        t1   = np.array(Image.open(t1p).convert("RGB"), dtype=np.float32) / 255.0
        t2   = np.array(Image.open(t2p).convert("RGB"), dtype=np.float32) / 255.0
        mask = (np.array(Image.open(mp).convert("L"), dtype=np.float32) > 127).astype(np.float32)
        h, w = t1.shape[:2]; cy, cx = h//2, w//2; hs = PATCH_SIZE//2
        if h < PATCH_SIZE or w < PATCH_SIZE: continue
        t1c = t1[cy-hs:cy+hs, cx-hs:cx+hs]; t2c = t2[cy-hs:cy+hs, cx-hs:cx+hs]
        mc  = mask[cy-hs:cy+hs, cx-hs:cx+hs]
        t1b = t1c[np.newaxis]; t2b = t2c[np.newaxis]
        if is_siamese:
            pred = model.predict({"t1_input": t1b, "t2_input": t2b}, verbose=0, batch_size=1)
        else:
            pred = model.predict(np.concatenate([t1b,t2b], axis=-1), verbose=0, batch_size=1)
        y_true_all.append(mc.flatten()); y_pred_all.append((pred[0,...,0]>0.5).astype(int).flatten())

    y_true = np.concatenate(y_true_all).astype(int)
    y_pred = np.concatenate(y_pred_all).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"{model_name} — Pixel Confusion Matrix", fontsize=12, fontweight="bold")
    tick_labels = ["No Change","Change"]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
                xticklabels=tick_labels, yticklabels=tick_labels, linewidths=2, linecolor="white", annot_kws={"size":13})
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual"); axes[0].set_title("Raw Counts")
    sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="RdYlGn", ax=axes[1],
                xticklabels=tick_labels, yticklabels=tick_labels, vmin=0, vmax=100,
                linewidths=2, linecolor="white", annot_kws={"size":13})
    axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual"); axes[1].set_title("Row-Normalised %")
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/11_confusion_{model_name.replace(' ','_')}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"{model_name}")
    print(classification_report(y_true, y_pred, target_names=["No Change","Change"]))

for model, name, siam in [(model_unet,"Baseline U-Net",False),
                           (model_dense,"DenseNet+UNet",False),
                           (model_siamese,"Siamese U-Net",True)]:
    plot_confusion_matrix(model, name, siam)

#Urban Expansion Trend Analysis

In [ ]:
def predict_change_pct(model, t1_path, t2_path, is_siamese=False):
    """Centre-crop full image to IMG_SIZE, infer, return % pixels changed."""
    t1 = np.array(Image.open(t1_path).convert("RGB"), dtype=np.float32) / 255.0
    t2 = np.array(Image.open(t2_path).convert("RGB"), dtype=np.float32) / 255.0
    h, w = t1.shape[:2]; cy, cx = h // 2, w // 2; hs = IMG_SIZE // 2
    t1 = t1[cy-hs:cy+hs, cx-hs:cx+hs]; t2 = t2[cy-hs:cy+hs, cx-hs:cx+hs]
    if t1.shape != (IMG_SIZE, IMG_SIZE, 3): return None
    t1b, t2b = t1[np.newaxis], t2[np.newaxis]
    if is_siamese:
        pred = model.predict({"t1_input": t1b, "t2_input": t2b}, verbose=0)[0, ..., 0]
    else:
        pred = model.predict(np.concatenate([t1b, t2b], axis=-1), verbose=0)[0, ..., 0]
    return 100.0 * (pred > 0.5).mean()


def plot_urban_expansion(model, model_name, analysis_paths, is_siamese=False):
    pcts = [v for t1p, t2p, _ in tqdm(analysis_paths, desc=f"expansion:{model_name}")
            if (v := predict_change_pct(model, t1p, t2p, is_siamese)) is not None]
    pcts = np.array(pcts)
    fig = plt.figure(figsize=(18, 11))
    fig.suptitle(f"{model_name} — Urban Expansion Analysis",
                 fontsize=14, fontweight="bold")
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, :2])
    spcts = np.sort(pcts)
    ax1.fill_between(range(len(spcts)), spcts, alpha=0.3, color="tomato")
    ax1.plot(spcts, color="crimson", linewidth=2.5)
    ax1.axhline(spcts.mean(),     color="navy",     ls="--", lw=2,
                label=f"Mean: {spcts.mean():.2f}%")
    ax1.axhline(np.median(spcts), color="darkcyan", ls=":",  lw=2,
                label=f"Median: {np.median(spcts):.2f}%")
    ax1.set_xlabel("Images sorted by change intensity"); ax1.set_ylabel("% Area Changed")
    ax1.set_title("Change Intensity Profile"); ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2 = fig.add_subplot(gs[0, 2])
    ax2.hist(pcts, bins=20, color="tomato", edgecolor="white", alpha=0.85)
    ax2.axvline(pcts.mean(), color="navy", ls="--", lw=2,
                label=f"Mean: {pcts.mean():.2f}%")
    ax2.set_xlabel("% Changed"); ax2.set_ylabel("Image Count")
    ax2.set_title("Change Distribution"); ax2.legend(); ax2.grid(True, alpha=0.3)

    ax3 = fig.add_subplot(gs[1, 0])
    ax3.pie(
        [(pcts < 5).sum(), ((pcts >= 5) & (pcts < 15)).sum(),
         ((pcts >= 15) & (pcts < 30)).sum(), (pcts >= 30).sum()],
        labels=["Minimal (<5%)", "Low (5-15%)", "High (15-30%)", "Severe (>30%)"],
        colors=["#4CAF50", "#FFC107", "#FF5722", "#B71C1C"],
        autopct="%1.1f%%", startangle=90,
        wedgeprops=dict(edgecolor="white", linewidth=2, width=0.65)
    )
    ax3.set_title("Severity Distribution")

    ax4 = fig.add_subplot(gs[1, 1:])
    ax4.axis("off")
    tbl = ax4.table(
        cellText=[
            ["Total images",        str(len(pcts))],
            ["Mean change",         f"{pcts.mean():.2f}%"],
            ["Median change",       f"{np.median(pcts):.2f}%"],
            ["Std",                 f"{pcts.std():.2f}%"],
            ["Min / Max",           f"{pcts.min():.2f}% / {pcts.max():.2f}%"],
            ["Images >10% change",  f"{(pcts>10).sum()} ({100*(pcts>10).mean():.1f}%)"],
            ["Images >25% change",  f"{(pcts>25).sum()} ({100*(pcts>25).mean():.1f}%)"],
        ],
        colLabels=["Metric", "Value"], cellLoc="center", loc="center",
        bbox=[0.05, 0.0, 0.9, 1.0]
    )
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.auto_set_column_width([0, 1])
    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor("#1565C0")
            cell.set_text_props(color="white", fontweight="bold")
        elif row % 2 == 0:
            cell.set_facecolor("#E3F2FD")
    ax4.set_title("Summary Statistics", fontsize=12, fontweight="bold", pad=10)
    plt.savefig(f"{PLOTS_DIR}/12_urban_expansion_{model_name.replace(' ', '_')}.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    return pcts

exp_unet    = plot_urban_expansion(model_unet,    "Baseline U-Net", test_paths)
exp_dense   = plot_urban_expansion(model_dense,   "DenseNet+UNet",  test_paths)
exp_siamese = plot_urban_expansion(model_siamese, "Siamese U-Net",  test_paths,
                                    is_siamese=True)

fig, ax = plt.subplots(figsize=(12, 5))
bp = ax.boxplot([exp_unet, exp_dense, exp_siamese], patch_artist=True, vert=True,
                widths=0.4, medianprops=dict(color="black", linewidth=2.5))
for patch, color in zip(bp["boxes"], list(COLORS.values())):
    patch.set_facecolor(color); patch.set_alpha(0.75)
ax.set_xticks([1, 2, 3]); ax.set_xticklabels(MODEL_LABELS, fontsize=12)
ax.set_ylabel("% Area Changed"); ax.set_title("Cross-Model Change Estimate Agreement")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/13_expansion_model_agreement.png", dpi=150, bbox_inches="tight")
plt.show()

#Cross-Dataset Evaluation (OSCD)

OSCD images are multi-band Sentinel-2 TIFs stored as individual files per band.
RGB is constructed from B04 (Red) + B03 (Green) + B02 (Blue) per the Sentinel-2 band naming.
Only the 14 cities that have ground-truth labels in train_labels are evaluated.

In [ ]:
OSCD_LABELLED_CITIES = [
    "abudhabi", "aguasclaras", "beihai", "beirut", "bercy",
    "bordeaux", "cupertino", "hongkong", "mumbai", "nantes",
    "paris", "pisa", "rennes", "saclay_e"
]


def load_oscd_rgb(city_img_dir):
    """
    OSCD stores one TIF per spectral band (B01.tif … B13.tif).
    Load B04 (Red), B03 (Green), B02 (Blue) and stack into an (H, W, 3) uint8 array.
    Falls back to pair/img1.png if individual band files are missing.
    """
    rgb_bands = []
    for band in ["B04", "B03", "B02"]:
        band_path = os.path.join(city_img_dir, f"{band}.tif")
        if not os.path.exists(band_path):
            # Some cities use different naming — try pair/img*.png fallback
            return None
        arr = np.array(Image.open(band_path))
        # Sentinel-2 L1C reflectance is uint16 in range ~0-10000; normalise to 0-255
        arr = np.clip(arr / 40.0, 0, 255).astype(np.uint8)
        rgb_bands.append(arr)
    return np.stack(rgb_bands, axis=-1)  # (H, W, 3)


def load_oscd_samples(images_root, labels_root, cities, img_size=256):
    """
    Returns list of {city, t1, t2, mask} dicts — one centre-cropped patch per city.
    T1/T2 are RGB arrays built from B04+B03+B02 bands.
    Mask is loaded from cm/ folder (png or tif, whichever exists).
    """
    samples = []
    for city in cities:
        t1_dir   = os.path.join(images_root, city, "imgs_1_rect")
        t2_dir   = os.path.join(images_root, city, "imgs_2_rect")
        mask_dir = os.path.join(labels_root, city, "cm")

        if not all(os.path.exists(d) for d in [t1_dir, t2_dir, mask_dir]):
            print(f"  skip {city}: missing dir"); continue

        t1 = load_oscd_rgb(t1_dir)
        t2 = load_oscd_rgb(t2_dir)

        # Fallback: if band files missing, use pair/img1.png and pair/img2.png
        if t1 is None or t2 is None:
            pair_dir = os.path.join(images_root, city, "pair")
            imgs = sorted(glob.glob(os.path.join(pair_dir, "img*.png")))
            if len(imgs) < 2:
                print(f"  skip {city}: no usable images"); continue
            t1 = np.array(Image.open(imgs[0]).convert("RGB"))
            t2 = np.array(Image.open(imgs[1]).convert("RGB"))

        # Load mask — try png then tif
        mask_files = (sorted(glob.glob(os.path.join(mask_dir, "*.png"))) +
                      sorted(glob.glob(os.path.join(mask_dir, "*.tif"))))
        if not mask_files:
            print(f"  skip {city}: no mask"); continue
        mask = np.array(Image.open(mask_files[0]).convert("L"))

        h = min(t1.shape[0], t2.shape[0], mask.shape[0])
        w = min(t1.shape[1], t2.shape[1], mask.shape[1])
        if h < img_size or w < img_size:
            print(f"  skip {city}: too small ({h}x{w})"); continue

        cy, cx = h // 2, w // 2; hs = img_size // 2
        samples.append({
            "city": city,
            "t1"  : t1  [cy-hs:cy+hs, cx-hs:cx+hs],
            "t2"  : t2  [cy-hs:cy+hs, cx-hs:cx+hs],
            "mask": mask[cy-hs:cy+hs, cx-hs:cx+hs],
        })

    print(f"OSCD: {len(samples)}/{len(cities)} cities loaded")
    return samples


oscd_samples = load_oscd_samples(OSCD_IMAGES_ROOT, OSCD_LABELS_ROOT,
                                  OSCD_LABELLED_CITIES)


def evaluate_on_oscd(model, samples, model_name, is_siamese=False):
    if not samples:
        print("No OSCD samples — skipping."); return pd.DataFrame()
    records = []
    for s in tqdm(samples, desc=f"OSCD:{model_name}"):
        t1 = s["t1"].astype(np.float32) / 255.0
        t2 = s["t2"].astype(np.float32) / 255.0
        mb = (s["mask"] > 127).astype(np.float32)
        t1b, t2b = t1[np.newaxis], t2[np.newaxis]
        if is_siamese:
            pred = model.predict({"t1_input": t1b, "t2_input": t2b},
                                 verbose=0)[0, ..., 0]
        else:
            pred = model.predict(np.concatenate([t1b, t2b], axis=-1),
                                 verbose=0)[0, ..., 0]
        pb   = (pred > 0.5).astype(np.float32)
        tp   = np.sum(mb * pb); fp = np.sum((1-mb)*pb); fn = np.sum(mb*(1-pb))
        iou  = tp / (tp + fp + fn + 1e-6)
        prec = tp / (tp + fp + 1e-6); rec = tp / (tp + fn + 1e-6)
        records.append({"city": s["city"], "iou": iou,
                         "f1": 2*prec*rec/(prec+rec+1e-6),
                         "precision": prec, "recall": rec})
    df = pd.DataFrame(records)
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle(f"{model_name} — OSCD Cross-Dataset Evaluation",
                 fontsize=13, fontweight="bold")
    bar_keys = ["iou", "f1", "precision", "recall"]
    bar_lbls = ["IoU", "F1", "Precision", "Recall"]
    bar_cols = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"]
    x = np.arange(len(df))
    for i, (k, lbl, c) in enumerate(zip(bar_keys, bar_lbls, bar_cols)):
        axes[0].bar(x + i * 0.2, df[k], 0.2, label=lbl, color=c, alpha=0.82)
    axes[0].set_xticks(x + 0.3)
    axes[0].set_xticklabels(df["city"], rotation=40, ha="right", fontsize=9)
    axes[0].set_ylim(0, 1.15); axes[0].set_ylabel("Score")
    axes[0].set_title("Per-City Metrics (OSCD)")
    axes[0].legend(); axes[0].grid(axis="y", alpha=0.3)

    levir_res = (results_unet  if "U-Net" in model_name and "Dense" not in model_name
                 else results_dense if "Dense" in model_name else results_siamese)
    oscd_mean = df[bar_keys].mean()
    lv_vals = [levir_res.get(k, 0) for k in bar_keys]
    os_vals = [oscd_mean[k] for k in bar_keys]
    xb = np.arange(len(bar_keys))
    axes[1].bar(xb - 0.2, lv_vals, 0.38, label="LEVIR-CD", color="steelblue", alpha=0.85)
    axes[1].bar(xb + 0.2, os_vals, 0.38, label="OSCD",     color="tomato",    alpha=0.85)
    for i, (lv, os) in enumerate(zip(lv_vals, os_vals)):
        axes[1].annotate(f"Δ{lv-os:+.2f}", xy=(i+0.2, os+0.02),
                         ha="center", fontsize=9, color="crimson", fontweight="bold")
    axes[1].set_xticks(xb); axes[1].set_xticklabels(bar_lbls, fontsize=11)
    axes[1].set_ylim(0, 1.2); axes[1].set_ylabel("Score")
    axes[1].set_title("Generalisation Gap: LEVIR-CD vs OSCD")
    axes[1].legend(); axes[1].grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/14_oscd_{model_name.replace(' ', '_')}.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\n{model_name} OSCD means:\n{df[bar_keys].mean().to_string()}")
    return df

oscd_unet    = evaluate_on_oscd(model_unet,    oscd_samples, "Baseline U-Net")
oscd_dense   = evaluate_on_oscd(model_dense,   oscd_samples, "DenseNet+UNet")
oscd_siamese = evaluate_on_oscd(model_siamese, oscd_samples, "Siamese U-Net",
                                 is_siamese=True)

#Save Everything

In [ ]:
for model, name in [(model_unet,    "baseline_unet"),
                     (model_dense,   "densenet_unet"),
                     (model_siamese, "siamese_unet")]:
    path = os.path.join(MODELS_DIR, f"{name}_final_weights.weights.h5")
    model.save_weights(path)
    print(f"weights -> {path}")

zip_path = os.path.join(OUTPUT_DIR, "urban_expansion_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder, label in [(PLOTS_DIR,  "plots"),
                           (LOGS_DIR,   "training_logs"),
                           (MODELS_DIR, "model_weights")]:
        for fp in sorted(glob.glob(os.path.join(folder, "*"))):
            zf.write(fp, f"{label}/{os.path.basename(fp)}")
print(f"zip saved -> {zip_path}")
print("All outputs are in the Kaggle Output tab.")